In [ ]:
# ============================================================
# BRANCH A — DIRECT DOCUMENT REPRESENTATION
# D4 — Eurostat LFS Metadata Workbook
# ============================================================
#
# Methodology stages covered:
# Stage 2 — Branch A: Direct Ingestion
# Stage 3 — Information Extraction using an LLM
# Post-extraction technical diagnostics
# ============================================================

from google.colab import files
from pathlib import Path

import hashlib
import json
import platform
import re
import sys

import pandas as pd

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D4"

DOCUMENT_NAME = (
    "Eurostat — LFS Metadata Excel — lfsa_esms"
)

BRANCH = "A"

BRANCH_NAME = "Direct Ingestion"

INPUT_REPRESENTATION = "Original XLSX workbook"

SOURCE_SHEET = "Metadata"

EXPECTED_SHEETS = [
    "Metadata",
    "Parameters",
    "Annexes"
]

EXPECTED_REFERENCE_SCOPE_ROWS = (
    list(range(2, 10))
    + list(range(12, 87))
)

EXPECTED_RECORD_COUNT = 83

EXPECTED_HEADER_RECORD_COUNT = 8

EXPECTED_CONCEPT_RECORD_COUNT = 75

EXPECTED_FIELDS = [
    "Section",
    "Concept Name",
    "Concept Value",
    "Publication Restricted"
]

ALLOWED_PUBLICATION_FLAGS = {
    "YES",
    "NO"
}

OUTPUT_DIR = Path(
    "outputs_D4_branch_A"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Branch name:", BRANCH_NAME)
print("Input representation:", INPUT_REPRESENTATION)
print("Expected records:", EXPECTED_RECORD_COUNT)
print("Output directory:", OUTPUT_DIR)

In [ ]:
# ============================================================
# 2. Source document upload
# ============================================================

print(
    "Upload the original D4 XLSX workbook."
)

uploaded = files.upload()

xlsx_files = [
    Path(filename)
    for filename in uploaded.keys()
    if filename.lower().endswith(".xlsx")
]

if len(xlsx_files) != 1:

    raise ValueError(
        "Upload exactly one XLSX workbook."
    )

WORKBOOK_PATH = xlsx_files[0]

print(
    "Uploaded workbook:",
    WORKBOOK_PATH.name
)

In [ ]:
# ============================================================
# 3. Source SHA-256
# ============================================================

def sha256_file(path):
    """
    Return the SHA-256 hash of a file.
    """

    hash_object = hashlib.sha256()

    with open(path, "rb") as file:

        for chunk in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b""
        ):

            hash_object.update(
                chunk
            )

    return hash_object.hexdigest()


WORKBOOK_SHA256 = sha256_file(
    WORKBOOK_PATH
)

print(
    "Workbook SHA-256:",
    WORKBOOK_SHA256
)

In [ ]:
# ============================================================
# 4. Workbook structure verification
# ============================================================

excel_file = pd.ExcelFile(
    WORKBOOK_PATH
)

sheet_names = (
    excel_file.sheet_names
)

expected_sheets_present = all(
    sheet_name in sheet_names
    for sheet_name in EXPECTED_SHEETS
)

unexpected_sheets = [
    sheet_name
    for sheet_name in sheet_names
    if sheet_name not in EXPECTED_SHEETS
]


print(
    "Workbook sheets:",
    sheet_names
)

print(
    "Expected sheets present:",
    expected_sheets_present
)

print(
    "Unexpected sheets:",
    unexpected_sheets
)


if not expected_sheets_present:

    raise ValueError(
        "The workbook does not contain all "
        "expected worksheets."
    )

In [ ]:
# ============================================================
# 5. Metadata worksheet diagnostics
# ============================================================

metadata_raw = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name=SOURCE_SHEET,
    header=None,
    dtype=object,
    keep_default_na=False
)


metadata_row_count = int(
    metadata_raw.shape[0]
)

metadata_column_count = int(
    metadata_raw.shape[1]
)


print(
    "Metadata rows:",
    metadata_row_count
)

print(
    "Metadata columns:",
    metadata_column_count
)

display(
    metadata_raw.head(
        20
    )
)

In [ ]:
# ============================================================
# 6. Metadata structure verification
# ============================================================

def clean_check_value(value):
    """
    Convert a worksheet cell to stripped text for
    integrity checking only.
    """

    if value is None:
        return ""

    return str(
        value
    ).strip()


metadata_first_column = [
    clean_check_value(
        value
    )
    for value in metadata_raw.iloc[
        :,
        0
    ].tolist()
]


structural_label_checks = {
    "row_1_header":
        (
            metadata_first_column[0]
            == "Header"
        ),

    "row_10_concepts":
        (
            metadata_first_column[9]
            == "Concepts"
        ),

    "row_11_concept_name":
        (
            metadata_first_column[10]
            == "Concept name"
        )
}


expected_row_count_valid = (
    metadata_row_count == 86
)

minimum_column_count_valid = (
    metadata_column_count >= 3
)

structural_labels_valid = all(
    structural_label_checks.values()
)


print(
    "Expected row count valid:",
    expected_row_count_valid
)

print(
    "Minimum column count valid:",
    minimum_column_count_valid
)

print(
    json.dumps(
        structural_label_checks,
        indent=2,
        ensure_ascii=False
    )
)

print(
    "Structural labels valid:",
    structural_labels_valid
)

In [ ]:
# ============================================================
# 7. Extraction-scope verification
# ============================================================

included_source_rows_present = all(
    worksheet_row
    <= metadata_row_count

    for worksheet_row
    in EXPECTED_REFERENCE_SCOPE_ROWS
)

expected_scope_row_count = len(
    EXPECTED_REFERENCE_SCOPE_ROWS
)

scope_row_count_valid = (
    expected_scope_row_count
    == EXPECTED_RECORD_COUNT
)


print(
    "Included source rows present:",
    included_source_rows_present
)

print(
    "Rows in fixed extraction scope:",
    expected_scope_row_count
)

print(
    "Scope count valid:",
    scope_row_count_valid
)

In [ ]:
# ============================================================
# 8. Source integrity report
# ============================================================

workbook_non_empty = (
    WORKBOOK_PATH.exists()
    and WORKBOOK_PATH.stat().st_size > 0
)

input_integrity_passed = all([
    workbook_non_empty,
    expected_sheets_present,
    expected_row_count_valid,
    minimum_column_count_valid,
    structural_labels_valid,
    included_source_rows_present,
    scope_row_count_valid
])


INPUT_INTEGRITY = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_file":
        WORKBOOK_PATH.name,

    "input_representation":
        INPUT_REPRESENTATION,

    "file_sha256":
        WORKBOOK_SHA256,

    "file_size_bytes":
        int(
            WORKBOOK_PATH.stat().st_size
        ),

    "sheet_names":
        sheet_names,

    "expected_sheets":
        EXPECTED_SHEETS,

    "expected_sheets_present":
        expected_sheets_present,

    "unexpected_sheets":
        unexpected_sheets,

    "source_sheet":
        SOURCE_SHEET,

    "metadata_row_count":
        metadata_row_count,

    "expected_metadata_row_count":
        86,

    "metadata_row_count_valid":
        expected_row_count_valid,

    "metadata_column_count":
        metadata_column_count,

    "minimum_column_count_valid":
        minimum_column_count_valid,

    "structural_label_checks":
        structural_label_checks,

    "structural_labels_valid":
        structural_labels_valid,

    "included_source_rows": (
        "Rows 2–9 and 12–86"
    ),

    "included_source_row_count":
        expected_scope_row_count,

    "included_source_rows_present":
        included_source_rows_present,

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "scope_row_count_valid":
        scope_row_count_valid,

    "direct_workbook_ingestion_usable":
        input_integrity_passed,

    "input_integrity_passed":
        input_integrity_passed
}


INPUT_INTEGRITY_PATH = (
    OUTPUT_DIR
    / "D4_branch_A_input_integrity.json"
)


with open(
    INPUT_INTEGRITY_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        INPUT_INTEGRITY,
        file,
        indent=2,
        ensure_ascii=False
    )


print(
    json.dumps(
        INPUT_INTEGRITY,
        indent=2,
        ensure_ascii=False
    )
)


if not input_integrity_passed:

    raise ValueError(
        "The original D4 workbook failed "
        "the Branch A integrity checks."
    )

In [ ]:
# ============================================================
# 9. Branch A representation
# ============================================================

BRANCH_REPRESENTATION = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "representation_type":
        "Original source document",

    "input_file":
        WORKBOOK_PATH.name,

    "input_format":
        WORKBOOK_PATH.suffix.lower(),

    "diagnostic_workbook_inspection_applied":
        True,

    "worksheet_to_text_conversion_applied":
        False,

    "derived_representation_used_as_model_input":
        False,

    "structural_conversion_applied":
        False,

    "normalisation_applied":
        False,

    "html_cleaning_applied":
        False,

    "worksheet_filtering_applied":
        False,

    "workbook_rewriting_applied":
        False,

    "model_input_description": (
        "The complete original XLSX workbook is submitted "
        "directly to the LLM. Workbook inspection with pandas "
        "is used only for source-integrity diagnostics. "
        "The extraction task restricts the target scope to "
        "Metadata rows 2–9 and 12–86, but the workbook itself "
        "is not filtered, rewritten or transformed."
    )
}


REPRESENTATION_PATH = (
    OUTPUT_DIR
    / "D4_branch_A_representation.json"
)


with open(
    REPRESENTATION_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        BRANCH_REPRESENTATION,
        file,
        indent=2,
        ensure_ascii=False
    )


print(
    json.dumps(
        BRANCH_REPRESENTATION,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 10. Extraction schema
# ============================================================

EXPECTED_OUTPUT_STRUCTURE = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "records": [
        {
            "Section": None,
            "Concept Name": None,
            "Concept Value": None,
            "Publication Restricted": None
        }
    ]
}


print(
    json.dumps(
        EXPECTED_OUTPUT_STRUCTURE,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 11. Fixed extraction task
# ============================================================

EXTRACTION_TASK = """
You are an information extraction assistant.

Extract every metadata record represented within the defined scope of
the Metadata worksheet of the attached original Excel workbook.

For every included record, extract:

- Section
- Concept Name
- Concept Value
- Publication Restricted

Scope rules:

- Treat the attached original Excel workbook as the only source of
  information.
- Use only the Metadata worksheet.
- Include the workbook-header metadata records in rows 2–9.
- Include every metadata concept record in rows 12–86.
- Include top-level metadata section rows even when their Concept Value
  is empty.
- Exclude row 1, labelled "Header".
- Exclude row 10, labelled "Concepts".
- Exclude row 11, containing the column headings "Concept name",
  "Concept value", and "Restricted from publication".
- Do not extract records from the Parameters worksheet.
- Do not extract records from the Annexes worksheet.

Extraction rules:

- Preserve Concept Name exactly as represented in the Metadata
  worksheet.
- Preserve Concept Value exactly as represented, including HTML-like
  tags, hyperlinks, attributes, entities, punctuation and internal line
  breaks.
- Do not render, simplify, remove or rewrite HTML-like content.
- Preserve YES and NO publication-restriction flags exactly.
- Use null when a Concept Value or Publication Restricted cell is
  genuinely empty.
- Assign workbook-header records from rows 2–9 to Section "Header".
- For numbered metadata concepts, use the complete top-level numbered
  section heading as Section, for example "1. Contact".
- Do not follow hyperlinks.
- Do not infer missing values.
- Do not calculate, summarise, paraphrase, translate, harmonise or
  correct source content.
- Do not use external knowledge.
- Return one record for every included worksheet row.
- Verify that only the defined Metadata worksheet scope has been
  processed.
- Verify that every record within that scope has been processed.
- Verify that empty source cells are represented as null.
- Verify that HTML-like source content remains unchanged.
- Return only valid JSON.
- Do not include Markdown fences, explanations or commentary.
- Keep the exact field names defined in the schema.
"""

In [ ]:
# ============================================================
# 12. Extraction prompt
# ============================================================

FULL_PROMPT = f"""
{EXTRACTION_TASK}

Expected JSON schema:
{json.dumps(
    EXPECTED_OUTPUT_STRUCTURE,
    indent=2,
    ensure_ascii=False
)}

The complete original XLSX workbook is attached as the extraction
source.

The extraction scope is restricted to Metadata rows 2–9 and 12–86.
The workbook itself has not been filtered, converted or transformed.

Return only the JSON object.
""".strip()


PROMPT_PATH = (
    OUTPUT_DIR
    / "D4_branch_A_prompt.txt"
)


PROMPT_PATH.write_text(
    FULL_PROMPT,
    encoding="utf-8"
)


PROMPT_SHA256 = hashlib.sha256(
    FULL_PROMPT.encode(
        "utf-8"
    )
).hexdigest()


print(
    FULL_PROMPT
)

print(
    "\nPrompt saved:",
    PROMPT_PATH
)

print(
    "Prompt SHA-256:",
    PROMPT_SHA256
)

In [ ]:
# ============================================================
# 13. Experiment metadata
# ============================================================

EXPERIMENT_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "source_file":
        WORKBOOK_PATH.name,

    "source_format":
        WORKBOOK_PATH.suffix.lower(),

    "source_sha256":
        WORKBOOK_SHA256,

    "source_structure": {
        "expected_sheets":
            EXPECTED_SHEETS,

        "observed_sheets":
            sheet_names,

        "expected_sheets_present":
            expected_sheets_present,

        "source_sheet":
            SOURCE_SHEET,

        "metadata_row_count":
            metadata_row_count,

        "metadata_row_count_verified":
            expected_row_count_valid,

        "metadata_column_count":
            metadata_column_count,

        "structural_labels_verified":
            structural_labels_valid,

        "fixed_extraction_scope_rows":
            "Metadata rows 2–9 and 12–86",

        "scope_rows_verified":
            included_source_rows_present
    },

    "input_representation":
        "Original XLSX workbook",

    "direct_document_ingestion":
        True,

    "diagnostic_workbook_inspection_applied":
        True,

    "worksheet_to_text_conversion_applied":
        False,

    "derived_representation_used_as_model_input":
        False,

    "structural_conversion_applied":
        False,

    "normalisation_applied":
        False,

    "html_cleaning_applied":
        False,

    "worksheet_filtering_applied":
        False,

    "workbook_rewriting_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "semantic_harmonisation_applied":
        False,

    "source_content_modification_applied":
        False,

    "expected_extraction_scope": {
        "expected_record_count":
            EXPECTED_RECORD_COUNT,

        "expected_header_record_count":
            EXPECTED_HEADER_RECORD_COUNT,

        "expected_concept_record_count":
            EXPECTED_CONCEPT_RECORD_COUNT,

        "expected_fields":
            EXPECTED_FIELDS
    },

    "content_included_in_extraction_task": [
        "Metadata rows 2–9",
        "Metadata rows 12–86"
    ],

    "content_excluded_from_extraction_task": [
        "Metadata row 1",
        "Metadata row 10",
        "Metadata row 11",
        "Parameters worksheet",
        "Annexes worksheet"
    ],

    "allowed_publication_flags": [
        "YES",
        "NO"
    ],

    "input_integrity_file":
        INPUT_INTEGRITY_PATH.name,

    "input_integrity_passed":
        bool(
            input_integrity_passed
        ),

    "representation_file":
        REPRESENTATION_PATH.name,

    "prompt_file":
        PROMPT_PATH.name,

    "prompt_sha256":
        PROMPT_SHA256,

    "expected_output_format":
        "JSON",

    "execution_environment":
        "Independent ChatGPT conversation",

    "notes": (
        "Branch A submits the complete original XLSX workbook "
        "directly to the model. Workbook inspection with pandas "
        "is used only for source-integrity diagnostics and is not "
        "supplied to the model. The fixed Stage 1 extraction task "
        "restricts the target scope to Metadata rows 2–9 and "
        "12–86. The workbook itself is not filtered, converted, "
        "cleaned, normalised or otherwise transformed."
    )
}


EXPERIMENT_METADATA_PATH = (
    OUTPUT_DIR
    / "D4_branch_A_experiment_metadata.json"
)


with open(
    EXPERIMENT_METADATA_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        EXPERIMENT_METADATA,
        file,
        indent=2,
        ensure_ascii=False
    )


print(
    json.dumps(
        EXPERIMENT_METADATA,
        indent=2,
        ensure_ascii=False
    )
)

## Independent Branch A extraction
In a new independent ChatGPT conversation.

Upload:

1. the complete original D4 XLSX workbook;
2. `D4_branch_A_prompt.txt`.

Paste the complete prompt and submit it.

Recommended filename:

`D4_branch_A_raw_response.txt`

Upload that file in the following notebook cell.

In [ ]:
# ============================================================
# 14. Raw response upload
# ============================================================

print(
    "Upload the file containing the complete "
    "unmodified D4 Branch A model response."
)

uploaded_output = files.upload()


if len(
    uploaded_output
) != 1:

    raise ValueError(
        "Upload exactly one file containing "
        "the complete raw D4 Branch A LLM response."
    )


RAW_OUTPUT_SOURCE_PATH = Path(
    next(
        iter(
            uploaded_output
        )
    )
)


print(
    "Uploaded raw response:",
    RAW_OUTPUT_SOURCE_PATH.name
)

In [ ]:
# ============================================================
# 15. Raw-response preservation
# ============================================================

RAW_RESPONSE_TEXT = (
    RAW_OUTPUT_SOURCE_PATH.read_text(
        encoding="utf-8"
    )
)


if not RAW_RESPONSE_TEXT.strip():

    raise ValueError(
        "The uploaded raw response is empty."
    )


RAW_RESPONSE_PATH = (
    OUTPUT_DIR
    / "D4_branch_A_raw_response.txt"
)


RAW_RESPONSE_PATH.write_text(
    RAW_RESPONSE_TEXT,
    encoding="utf-8"
)


RAW_RESPONSE_SHA256 = hashlib.sha256(
    RAW_RESPONSE_TEXT.encode(
        "utf-8"
    )
).hexdigest()


print(
    "Raw response preserved:",
    RAW_RESPONSE_PATH
)

print(
    "Raw response SHA-256:",
    RAW_RESPONSE_SHA256
)

In [ ]:
# ============================================================
# 16. Raw-response parsing
# ============================================================

valid_json = True

json_parsing_error = None

PARSED_EXTRACTION = None


try:

    PARSED_EXTRACTION = json.loads(
        RAW_RESPONSE_TEXT
    )


except json.JSONDecodeError as error:

    valid_json = False

    json_parsing_error = str(
        error
    )


print(
    "Valid JSON:",
    valid_json
)

print(
    "JSON parsing error:",
    json_parsing_error
)

In [ ]:
# ============================================================
# 17. Top-level structure diagnostics
# ============================================================

top_level_object_valid = False

document_id_present = False
document_id_correct = False

branch_present = False
branch_correct = False

records_present = False
records_is_list = False

records_evaluable = False

extracted_records = []


if (
    valid_json
    and isinstance(
        PARSED_EXTRACTION,
        dict
    )
):

    top_level_object_valid = True

    document_id_present = (
        "document_id"
        in PARSED_EXTRACTION
    )

    document_id_correct = (
        PARSED_EXTRACTION.get(
            "document_id"
        )
        == DOCUMENT_ID
    )

    branch_present = (
        "branch"
        in PARSED_EXTRACTION
    )

    branch_correct = (
        PARSED_EXTRACTION.get(
            "branch"
        )
        == BRANCH
    )

    records_present = (
        "records"
        in PARSED_EXTRACTION
    )

    records_is_list = isinstance(
        PARSED_EXTRACTION.get(
            "records"
        ),
        list
    )

    if records_is_list:

        extracted_records = (
            PARSED_EXTRACTION[
                "records"
            ]
        )


records_evaluable = (
    valid_json
    and top_level_object_valid
    and records_present
    and records_is_list
)


number_of_records = len(
    extracted_records
)


record_count_valid = (
    number_of_records
    == EXPECTED_RECORD_COUNT
    if records_evaluable
    else None
)


print(
    "Top-level object valid:",
    top_level_object_valid
)

print(
    "Document ID present:",
    document_id_present
)

print(
    "Document ID correct:",
    document_id_correct
)

print(
    "Branch present:",
    branch_present
)

print(
    "Branch correct:",
    branch_correct
)

print(
    "Records present:",
    records_present
)

print(
    "Records is list:",
    records_is_list
)

print(
    "Records evaluable:",
    records_evaluable
)

print(
    "Observed records:",
    (
        number_of_records
        if records_evaluable
        else None
    )
)

print(
    "Expected records:",
    EXPECTED_RECORD_COUNT
)

print(
    "Record count valid:",
    record_count_valid
)

In [ ]:
# ============================================================
# 18. Parsed extraction
# ============================================================

PARSED_EXTRACTION_PATH = (
    OUTPUT_DIR
    / "D4_branch_A_parsed_extraction.json"
)


if valid_json:

    with open(
        PARSED_EXTRACTION_PATH,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            PARSED_EXTRACTION,
            file,
            indent=2,
            ensure_ascii=False
        )

    print(
        "Parsed extraction saved:",
        PARSED_EXTRACTION_PATH
    )


else:

    print(
        "Parsed extraction was not created because "
        "the raw response is not valid JSON."
    )

In [ ]:
# ============================================================
# 19. Record-structure diagnostics
# ============================================================

record_structure_issues = []

for record_index, record in enumerate(
    extracted_records
):

    issues = []

    if not isinstance(
        record,
        dict
    ):

        issues.append(
            "Record is not a JSON object."
        )

    else:

        actual_fields = set(
            record.keys()
        )

        expected_fields = set(
            EXPECTED_FIELDS
        )

        missing_fields = sorted(
            expected_fields
            - actual_fields
        )

        extra_fields = sorted(
            actual_fields
            - expected_fields
        )

        if missing_fields:

            issues.append({
                "missing_fields":
                    missing_fields
            })

        if extra_fields:

            issues.append({
                "extra_fields":
                    extra_fields
            })

    if issues:

        record_structure_issues.append({
            "record_index":
                record_index,

            "issues":
                issues
        })


records_with_structure_issues = len(
    record_structure_issues
)


print(
    "Records with structure issues:",
    records_with_structure_issues
)

In [ ]:
# ============================================================
# 20. Field-type diagnostics
# ============================================================

field_type_issues = []

for record_index, record in enumerate(
    extracted_records
):

    if not isinstance(
        record,
        dict
    ):

        continue

    for field in [
        "Section",
        "Concept Name",
        "Concept Value",
        "Publication Restricted"
    ]:

        value = record.get(
            field
        )

        if (
            value is not None
            and not isinstance(
                value,
                str
            )
        ):

            field_type_issues.append({
                "record_index":
                    record_index,

                "field":
                    field,

                "observed_type":
                    type(
                        value
                    ).__name__
            })


records_with_type_issues = len({
    issue[
        "record_index"
    ]
    for issue
    in field_type_issues
})


print(
    "Records with type issues:",
    records_with_type_issues
)

In [ ]:
# ============================================================
# 21. Publication-flag diagnostics
# ============================================================

publication_flag_issues = []

for record_index, record in enumerate(
    extracted_records
):

    if not isinstance(
        record,
        dict
    ):

        continue

    observed_flag = record.get(
        "Publication Restricted"
    )

    if (
        observed_flag is not None
        and observed_flag
        not in ALLOWED_PUBLICATION_FLAGS
    ):

        publication_flag_issues.append({
            "record_index":
                record_index,

            "concept_name":
                record.get(
                    "Concept Name"
                ),

            "observed_flag":
                observed_flag
        })


publication_flags_valid = (
    len(
        publication_flag_issues
    )
    == 0
)

In [ ]:
# ============================================================
# 22. Missing-value diagnostics
# ============================================================

missing_values_by_field = {
    field: sum(
        1

        for record
        in extracted_records

        if (
            not isinstance(
                record,
                dict
            )
            or record.get(
                field
            ) is None
        )
    )

    for field
    in EXPECTED_FIELDS
}


mandatory_field_missing_count = (
    missing_values_by_field[
        "Section"
    ]
    + missing_values_by_field[
        "Concept Name"
    ]
)


print(
    "Missing values by field:"
)

print(
    json.dumps(
        missing_values_by_field,
        indent=2,
        ensure_ascii=False
    )
)

print(
    "Missing mandatory values:",
    mandatory_field_missing_count
)

In [ ]:
# ============================================================
# 23. Record-count diagnostics
# ============================================================

header_record_count = sum(
    1

    for record
    in extracted_records

    if (
        isinstance(
            record,
            dict
        )
        and record.get(
            "Section"
        )
        == "Header"
    )
)


concept_record_count = (
    number_of_records
    - header_record_count
)


header_record_count_valid = (
    header_record_count
    == EXPECTED_HEADER_RECORD_COUNT
)

concept_record_count_valid = (
    concept_record_count
    == EXPECTED_CONCEPT_RECORD_COUNT
)


print(
    "Header records:",
    header_record_count
)

print(
    "Concept records:",
    concept_record_count
)

print(
    "Header count valid:",
    header_record_count_valid
)

print(
    "Concept count valid:",
    concept_record_count_valid
)

In [ ]:
# ============================================================
# 24. Duplicate-record diagnostics
# ============================================================

def create_record_key(record):
    """
    Create a strict extracted-record key.
    """

    if not isinstance(
        record,
        dict
    ):

        return None

    return (
        record.get(
            "Section"
        ),
        record.get(
            "Concept Name"
        )
    )


record_keys = [
    create_record_key(
        record
    )
    for record
    in extracted_records
]


duplicate_record_keys = sorted(
    {
        key

        for key
        in record_keys

        if (
            key is not None
            and record_keys.count(
                key
            ) > 1
        )
    },
    key=lambda value: str(
        value
    )
)


print(
    "Duplicate record-key count:",
    len(
        duplicate_record_keys
    )
)

In [ ]:
# ============================================================
# 25. Excluded-content diagnostics
# ============================================================

EXCLUDED_CONCEPT_NAMES = {
    "Header",
    "Concepts",
    "Concept name"
}


excluded_content_issues = []

for record_index, record in enumerate(
    extracted_records
):

    if not isinstance(
        record,
        dict
    ):

        continue

    concept_name = record.get(
        "Concept Name"
    )

    if concept_name in (
        EXCLUDED_CONCEPT_NAMES
    ):

        excluded_content_issues.append({
            "record_index":
                record_index,

            "concept_name":
                concept_name
        })


print(
    "Excluded-content issues:",
    len(
        excluded_content_issues
    )
)

In [ ]:
# ============================================================
# 26. HTML-representation diagnostics
# ============================================================

html_tag_pattern = re.compile(
    r"<[^>]+>"
)

html_entity_pattern = re.compile(
    r"&(?:nbsp|amp|lt|gt|quot|apos);",
    flags=re.IGNORECASE
)


records_with_html_tags = 0

records_with_html_entities = 0

for record in extracted_records:

    if not isinstance(
        record,
        dict
    ):

        continue

    concept_value = record.get(
        "Concept Value"
    )

    if not isinstance(
        concept_value,
        str
    ):

        continue

    if html_tag_pattern.search(
        concept_value
    ):

        records_with_html_tags += 1

    if html_entity_pattern.search(
        concept_value
    ):

        records_with_html_entities += 1


print(
    "Extracted values with HTML tags:",
    records_with_html_tags
)

print(
    "Extracted values with HTML entities:",
    records_with_html_entities
)

In [ ]:
# ============================================================
# 27. Structural evaluability
# ============================================================

structurally_evaluable = all([
    valid_json,
    top_level_object_valid,
    document_id_present,
    document_id_correct,
    branch_present,
    branch_correct,
    records_present,
    records_is_list,
    records_evaluable,
    records_with_structure_issues == 0,
    records_with_type_issues == 0
])


print(
    "Valid JSON:",
    valid_json
)

print(
    "Records evaluable:",
    records_evaluable
)

print(
    "Record count valid:",
    record_count_valid
)

print(
    "Structurally evaluable:",
    structurally_evaluable
)

In [ ]:
# ============================================================
# 28. Technical diagnostic summary
# ============================================================

TECHNICAL_DIAGNOSTICS = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "valid_json":
        valid_json,

    "json_parsing_error":
        json_parsing_error,

    "top_level_object_valid":
        top_level_object_valid,

    "document_id_present":
        document_id_present,

    "document_id_correct":
        document_id_correct,

    "branch_present":
        branch_present,

    "branch_correct":
        branch_correct,

    "records_present":
        records_present,

    "records_is_list":
        records_is_list,

    "records_evaluable":
        records_evaluable,

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "number_of_records":
        (
            number_of_records
            if records_evaluable
            else None
        ),

    "record_count_valid":
        (
            record_count_valid
            if records_evaluable
            else None
        ),

    "expected_header_record_count":
        EXPECTED_HEADER_RECORD_COUNT,

    "observed_header_record_count":
        (
            header_record_count
            if records_evaluable
            else None
        ),

    "header_record_count_valid":
        (
            header_record_count_valid
            if records_evaluable
            else None
        ),

    "expected_concept_record_count":
        EXPECTED_CONCEPT_RECORD_COUNT,

    "observed_concept_record_count":
        (
            concept_record_count
            if records_evaluable
            else None
        ),

    "concept_record_count_valid":
        (
            concept_record_count_valid
            if records_evaluable
            else None
        ),

    "records_with_structure_issues":
        (
            records_with_structure_issues
            if records_evaluable
            else None
        ),

    "record_structure_issues":
        (
            record_structure_issues
            if records_evaluable
            else None
        ),

    "records_with_type_issues":
        (
            records_with_type_issues
            if records_evaluable
            else None
        ),

    "field_type_issues":
        (
            field_type_issues
            if records_evaluable
            else None
        ),

    "publication_flags_valid":
        (
            publication_flags_valid
            if records_evaluable
            else None
        ),

    "publication_flag_issue_count":
        (
            len(
                publication_flag_issues
            )
            if records_evaluable
            else None
        ),

    "publication_flag_issues":
        (
            publication_flag_issues
            if records_evaluable
            else None
        ),

    "missing_values_by_field":
        (
            missing_values_by_field
            if records_evaluable
            else None
        ),

    "missing_mandatory_value_count":
        (
            mandatory_field_missing_count
            if records_evaluable
            else None
        ),

    "duplicate_record_key_count":
        (
            len(
                duplicate_record_keys
            )
            if records_evaluable
            else None
        ),

    "duplicate_record_keys":
        (
            duplicate_record_keys
            if records_evaluable
            else None
        ),

    "excluded_content_issue_count":
        (
            len(
                excluded_content_issues
            )
            if records_evaluable
            else None
        ),

    "excluded_content_issues":
        (
            excluded_content_issues
            if records_evaluable
            else None
        ),

    "records_with_html_tags":
        (
            records_with_html_tags
            if records_evaluable
            else None
        ),

    "records_with_html_entities":
        (
            records_with_html_entities
            if records_evaluable
            else None
        ),

    "structurally_evaluable":
        bool(structurally_evaluable)
}


TECHNICAL_DIAGNOSTICS_PATH = (
    OUTPUT_DIR
    / "D4_branch_A_technical_diagnostics.json"
)


with open(
    TECHNICAL_DIAGNOSTICS_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        TECHNICAL_DIAGNOSTICS,
        file,
        indent=2,
        ensure_ascii=False
    )


print(
    json.dumps(
        TECHNICAL_DIAGNOSTICS,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 29. Experiment metadata update
# ============================================================

EXPERIMENT_METADATA.update({
    "raw_response_file":
        RAW_RESPONSE_PATH.name,

    "raw_response_sha256":
        RAW_RESPONSE_SHA256,

    "structurally_evaluable":
        bool(structurally_evaluable),

    "parsed_extraction_file":
        (
            PARSED_EXTRACTION_PATH.name
            if valid_json
            else None
        ),

    "json_valid":
        valid_json,

    "records_evaluable":
        records_evaluable,

    "observed_record_count":
        (
            number_of_records
            if records_evaluable
            else None
        )
})


with open(
    EXPERIMENT_METADATA_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        EXPERIMENT_METADATA,
        file,
        indent=2,
        ensure_ascii=False
    )


print(
    "Experiment metadata updated with "
    "post-extraction information."
)

In [ ]:
# ============================================================
# 30. Experiment summary
# ============================================================

EXPERIMENT_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "source_file":
        WORKBOOK_PATH.name,

    "source_sha256":
        WORKBOOK_SHA256,

    "source_verified":
        bool(
            input_integrity_passed
        ),

    "input_representation":
        "Original XLSX workbook",

    "direct_document_ingestion":
        True,

    "diagnostic_workbook_inspection_applied":
        True,

    "worksheet_to_text_conversion_applied":
        False,

    "structural_conversion_applied":
        False,

    "normalisation_applied":
        False,

    "html_cleaning_applied":
        False,

    "worksheet_filtering_applied":
        False,

    "derived_representation_used_as_model_input":
        False,

    "json_valid":
        bool(
            valid_json
        ),

    "records_evaluable":
        bool(
            records_evaluable
        ),

    "structurally_evaluable":
        bool(structurally_evaluable),

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        (
            number_of_records
            if records_evaluable
            else None
        ),

    "record_count_matches":
        (
            bool(
                record_count_valid
            )
            if records_evaluable
            else None
        ),

    "expected_header_record_count":
        EXPECTED_HEADER_RECORD_COUNT,

    "observed_header_record_count":
        (
            header_record_count
            if records_evaluable
            else None
        ),

    "header_record_count_matches":
        (
            bool(
                header_record_count_valid
            )
            if records_evaluable
            else None
        ),

    "expected_concept_record_count":
        EXPECTED_CONCEPT_RECORD_COUNT,

    "observed_concept_record_count":
        (
            concept_record_count
            if records_evaluable
            else None
        ),

    "concept_record_count_matches":
        (
            bool(
                concept_record_count_valid
            )
            if records_evaluable
            else None
        ),

    "records_with_structure_issues":
        (
            records_with_structure_issues
            if records_evaluable
            else None
        ),

    "records_with_type_issues":
        (
            records_with_type_issues
            if records_evaluable
            else None
        ),

    "publication_flag_issue_count":
        (
            len(
                publication_flag_issues
            )
            if records_evaluable
            else None
        ),

    "missing_mandatory_value_count":
        (
            mandatory_field_missing_count
            if records_evaluable
            else None
        ),

    "duplicate_record_key_count":
        (
            len(
                duplicate_record_keys
            )
            if records_evaluable
            else None
        ),

    "excluded_content_issue_count":
        (
            len(
                excluded_content_issues
            )
            if records_evaluable
            else None
        ),

    "records_with_html_tags":
        (
            records_with_html_tags
            if records_evaluable
            else None
        ),

    "records_with_html_entities":
        (
            records_with_html_entities
            if records_evaluable
            else None
        ),

    "raw_response_preserved":
        RAW_RESPONSE_PATH.exists(),

    "parsed_extraction_created":
        bool(
            valid_json
        ),

    "content_validation_performed":
        False,

    "notes": (
        "This notebook performs source verification, "
        "D4 Branch A direct-workbook execution preservation "
        "and technical output checks only. Agreement with the "
        "fixed Stage 1 reference dataset is evaluated in the "
        "separate Validation A — D4 notebook."
    )
}


EXPERIMENT_SUMMARY_PATH = (
    OUTPUT_DIR
    / "D4_branch_A_experiment_summary.json"
)


with open(
    EXPERIMENT_SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        EXPERIMENT_SUMMARY,
        file,
        indent=2,
        ensure_ascii=False
    )


print(
    json.dumps(
        EXPERIMENT_SUMMARY,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 31. Display final experiment summary
# ============================================================

observed_records_display = (
    number_of_records
    if records_evaluable
    else "Not evaluable"
)

record_count_display = (
    record_count_valid
    if records_evaluable
    else "Not evaluable"
)

header_count_display = (
    header_record_count
    if records_evaluable
    else "Not evaluable"
)

concept_count_display = (
    concept_record_count
    if records_evaluable
    else "Not evaluable"
)

structure_issues_display = (
    records_with_structure_issues
    if records_evaluable
    else "Not evaluable"
)

type_issues_display = (
    records_with_type_issues
    if records_evaluable
    else "Not evaluable"
)

publication_issues_display = (
    len(
        publication_flag_issues
    )
    if records_evaluable
    else "Not evaluable"
)

duplicate_keys_display = (
    len(
        duplicate_record_keys
    )
    if records_evaluable
    else "Not evaluable"
)

excluded_content_display = (
    len(
        excluded_content_issues
    )
    if records_evaluable
    else "Not evaluable"
)


print(
    "=" * 48
)

print(
    "D4 Branch A experiment completed"
)

print(
    "=" * 48
)

print(
    f"Expected records        : "
    f"{EXPECTED_RECORD_COUNT}"
)

print(
    f"Observed records        : "
    f"{observed_records_display}"
)

print(
    f"Valid JSON              : "
    f"{valid_json}"
)

print(
    f"Records evaluable       : "
    f"{records_evaluable}"
)

print(
    f"Record count matches    : "
    f"{record_count_display}"
)

print(
    f"Structurally evaluable  : "
    f"{structurally_evaluable}"
)

print(
    f"Header records          : "
    f"{header_count_display}"
)

print(
    f"Concept records         : "
    f"{concept_count_display}"
)

print(
    f"Structure issues        : "
    f"{structure_issues_display}"
)

print(
    f"Type issues             : "
    f"{type_issues_display}"
)

print(
    f"Publication flag issues : "
    f"{publication_issues_display}"
)

print(
    f"Duplicate keys          : "
    f"{duplicate_keys_display}"
)

print(
    f"Excluded-content issues : "
    f"{excluded_content_display}"
)

print()

print(
    "Content validation performed: False"
)

print(
    "Next step: Validation A — D4"
)

In [ ]:
# ============================================================
# 32. Final artefact inventory
# ============================================================

GENERATED_OUTPUTS = [
    INPUT_INTEGRITY_PATH,
    REPRESENTATION_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PATH,
    RAW_RESPONSE_PATH,
    TECHNICAL_DIAGNOSTICS_PATH,
    EXPERIMENT_SUMMARY_PATH
]


if valid_json:

    GENERATED_OUTPUTS.insert(
        5,
        PARSED_EXTRACTION_PATH
    )


print(
    "Generated D4 Branch A files:\n"
)


for output_path in GENERATED_OUTPUTS:

    print(
        "-",
        output_path.name,
        "| exists:",
        output_path.exists()
    )

In [ ]:
# ============================================================
# 33. Download experiment artefacts
# ============================================================

for output_path in GENERATED_OUTPUTS:

    if output_path.exists():

        files.download(
            str(
                output_path
            )
        )